In [ ]:
import sys
print(f"Python: {sys.executable}")
print(f"Version: {sys.version.split()[0]}")

import statsmodels
import pandas as pd
print(f"statsmodels: {statsmodels.__version__}")
print(f"pandas: {pd.__version__}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Load the weekly cashflow target constructed on Day 4 of Phase 2
cashflow = pd.read_parquet("../data/processed/weekly_cashflow.parquet")

print(f"Weekly cashflow target loaded:")
print(f"  Shape:            {cashflow.shape}")
print(f"  Date range:       {cashflow.index.min().date()} → {cashflow.index.max().date()}")
print(f"  Columns:          {list(cashflow.columns)}")
print(f"  Index type:       {type(cashflow.index).__name__}")
print(f"\nFirst 3 rows:")
print(cashflow.head(3))

In [ ]:
# Calendar / time features
# ------------------------
# These extract signal from the date itself. Six features:
#   - year:            captures long-term growth trend
#   - quarter:         Q1-Q4 seasonality
#   - month:           monthly cycle
#   - week_of_year:    finer weekly cycle
#   - week_of_month:   month-end vs month-start effects
#   - is_quarter_end:  binary flag for Q-end weeks (often high activity)

features = cashflow.copy()  # keep original safe; work on a copy

features['year']           = features.index.year
features['quarter']        = features.index.quarter
features['month']          = features.index.month
features['week_of_year']   = features.index.isocalendar().week.astype(int)
features['week_of_month']  = (features.index.day - 1) // 7 + 1
features['is_quarter_end'] = features.index.month.isin([3, 6, 9, 12]).astype(int)

print("Calendar features added.")
print(f"Feature columns now: {[c for c in features.columns if c not in cashflow.columns]}")
print(f"\nSample (first 5 rows):")
print(features[['year', 'quarter', 'month', 'week_of_year', 'week_of_month', 'is_quarter_end']].head())

In [ ]:
# Lag features — value from N weeks ago
# --------------------------------------
# Positive shift is CORRECT: .shift(N) moves values DOWN by N rows,
# so row 100's lag_4 column contains the value that was at row 96.
# Negative shift would leak the future into the past — never do that.

LAGS = [1, 2, 4, 8, 12, 26, 52]
BASE_SERIES = ['sales', 'purchases', 'net_cashflow']

for series in BASE_SERIES:
    for lag in LAGS:
        features[f'{series}_lag_{lag}'] = features[series].shift(lag)

# Show which columns were created
lag_cols = [c for c in features.columns if '_lag_' in c]
print(f"Lag features created: {len(lag_cols)}")
print(f"  Series × Lags = {len(BASE_SERIES)} × {len(LAGS)} = {len(BASE_SERIES) * len(LAGS)}")

# Verify no future leakage — lag_1 at row N should equal the base value at row N-1
print(f"\nLeakage check (sales_lag_1 at row 5 should equal sales at row 4):")
print(f"  sales at row 4:        {features['sales'].iloc[4]:>12,.2f}")
print(f"  sales_lag_1 at row 5:  {features['sales_lag_1'].iloc[5]:>12,.2f}")
match = features['sales'].iloc[4] == features['sales_lag_1'].iloc[5]
print(f"  Match: {match}")

# How many NaN rows we've introduced (from the largest lag)
print(f"\nNaN count in sales_lag_52 (largest lag): {features['sales_lag_52'].isnull().sum()}")
print(f"  Expected: 52 (first 52 rows have no 52-weeks-ago value)")

In [ ]:
# Rolling aggregate features — recent-window statistics
# ------------------------------------------------------
# CRITICAL: .shift(1) is applied BEFORE .rolling() to prevent leakage.
# Without shift(1), the rolling window at row N would include row N itself
# (the target we're trying to predict) — silent leakage.

ROLLING_WINDOWS = [4, 12, 26]
ROLLING_STATS   = ['mean', 'std', 'sum']

for series in BASE_SERIES:
    shifted = features[series].shift(1)  # shift once, reuse for all windows

    for window in ROLLING_WINDOWS:
        rolling_obj = shifted.rolling(window=window, min_periods=window)

        features[f'{series}_roll{window}w_mean'] = rolling_obj.mean()
        features[f'{series}_roll{window}w_std']  = rolling_obj.std()
        features[f'{series}_roll{window}w_sum']  = rolling_obj.sum()

# Confirm what was created
roll_cols = [c for c in features.columns if '_roll' in c]
print(f"Rolling features created: {len(roll_cols)}")
print(f"  Series × Windows × Stats = 3 × 3 × 3 = 27")

# Leakage verification — the 4-week rolling mean at row 10 should equal
# the mean of the raw values at rows 6, 7, 8, 9 (NOT rows 7, 8, 9, 10)
print(f"\nLeakage check (sales_roll4w_mean at row 10):")
raw_window   = features['sales'].iloc[6:10]     # rows 6, 7, 8, 9 (correct)
wrong_window = features['sales'].iloc[7:11]     # rows 7, 8, 9, 10 (would be leakage)
feature_val  = features['sales_roll4w_mean'].iloc[10]

print(f"  Feature value:              {feature_val:>12,.2f}")
print(f"  Correct  (rows 6-9 mean):   {raw_window.mean():>12,.2f}")
print(f"  Leaky    (rows 7-10 mean):  {wrong_window.mean():>12,.2f}")
print(f"  Feature matches correct?    {np.isclose(feature_val, raw_window.mean())}")
print(f"  Feature matches leaky?      {np.isclose(feature_val, wrong_window.mean())} (should be False)")

# Where NaNs sit
print(f"\nNaN count in longest-window feature (net_cashflow_roll26w_std): "
      f"{features['net_cashflow_roll26w_std'].isnull().sum()}")
print(f"  Expected: ~26 (shift(1) + rolling(26) needs 26 prior values)")

print(f"\nCurrent feature count: {features.shape[1]} columns")

In [ ]:
# Save features and summarise
# ---------------------------
os.makedirs("../data/processed", exist_ok=True)
features.to_parquet("../data/processed/features_v1.parquet")

# Summary
print("=" * 60)
print("FEATURE ENGINEERING SUMMARY")
print("=" * 60)

feature_families = {
    'Base target columns':  [c for c in features.columns if c in BASE_SERIES],
    'Calendar features':    [c for c in features.columns 
                             if c in ['year','quarter','month','week_of_year',
                                      'week_of_month','is_quarter_end']],
    'Lag features':         [c for c in features.columns if '_lag_' in c],
    'Rolling features':     [c for c in features.columns if '_roll' in c],
}

for family, cols in feature_families.items():
    print(f"\n{family}: {len(cols)}")
    for c in cols[:5]:
        print(f"  {c}")
    if len(cols) > 5:
        print(f"  ... and {len(cols) - 5} more")

print(f"\n{'='*60}")
print(f"Total columns:            {features.shape[1]}")
print(f"Total rows (weeks):       {features.shape[0]}")
print(f"Rows with any NaN:        {features.isnull().any(axis=1).sum()}")
print(f"Rows with NO NaN:         {(~features.isnull().any(axis=1)).sum()}")
print(f"Fully-usable date range:  "
      f"{features.dropna().index.min().date()} → "
      f"{features.dropna().index.max().date()}")
print(f"\nSaved: ../data/processed/features_v1.parquet")

In [ ]:
# Payment-cycle features — dissertation-specific engineered features
# ------------------------------------------------------------------
# Based on the SME partner's documented payment behaviour:
#   30-day terms + 3-7 day processing → invoices paid 4-5 weeks after issue
#   45-day terms + 3-7 day processing → invoices paid 6-7 weeks after issue
#   60-day terms + 3-7 day processing → invoices paid 8-9 weeks after issue
#   90-day terms + 3-7 day processing → invoices paid 12-14 weeks after issue (larger invoices)
#
# Each feature at week t = sum of invoices issued in the relevant historical window
# that are most likely to convert to cash around week t.

# Windows defined as (start_week_lag, end_week_lag) — both inclusive.
# The rolling window is applied to the SHIFTED base series to prevent
# any leakage of week t itself into the feature.

PAY_WINDOWS = {
    '30d':  (4, 5),    # 4-5 weeks ago  → 28-35 days
    '45d':  (6, 7),    # 6-7 weeks ago  → 42-49 days
    '60d':  (8, 9),    # 8-9 weeks ago  → 56-63 days
    '90d':  (12, 14),  # 12-14 weeks ago → 84-98 days (larger invoices)
    'full': (2, 13),   # 2-13 weeks ago → ~14-91 days (aggregate view)
}

for series in ['sales', 'purchases']:
    base = features[series]

    for label, (start_lag, end_lag) in PAY_WINDOWS.items():
        # Sum of values from start_lag to end_lag weeks ago (inclusive)
        # Achieved by: shift by start_lag, then rolling window of (end_lag - start_lag + 1)
        window_size = end_lag - start_lag + 1
        shifted     = base.shift(start_lag)
        features[f'{series}_paywindow_{label}'] = shifted.rolling(
            window=window_size, min_periods=window_size
        ).sum()

# Derived net position and ratio features
features['expected_net_position']   = (features['sales_paywindow_full'] 
                                       - features['purchases_paywindow_full'])
features['sales_to_purchase_ratio'] = (features['sales_paywindow_full'] 
                                       / (features['purchases_paywindow_full'] + 1))

# Confirm what was created
paycycle_cols = [c for c in features.columns 
                 if 'paywindow' in c or c in ['expected_net_position', 'sales_to_purchase_ratio']]
print(f"Payment-cycle features created: {len(paycycle_cols)}")
for c in paycycle_cols:
    print(f"  {c}")

# Leakage check — sales_paywindow_30d at row 20 should equal sum of raw sales at rows 15-16
# (since window is lag 4-5 → uses values 4 and 5 rows earlier)
print(f"\nLeakage check (sales_paywindow_30d at row 20):")
correct_sum = features['sales'].iloc[15:17].sum()  # rows 15 and 16 (lags 5 and 4)
feature_val = features['sales_paywindow_30d'].iloc[20]
print(f"  Feature value:                  {feature_val:>12,.2f}")
print(f"  Correct (rows 15-16 sum):       {correct_sum:>12,.2f}")
print(f"  Match: {np.isclose(feature_val, correct_sum)}")

# Sanity check — expected_net_position should equal sales_full minus purchases_full
enp_check = (features['sales_paywindow_full'].iloc[100] 
             - features['purchases_paywindow_full'].iloc[100])
print(f"\nDerived feature verification (expected_net_position at row 100):")
print(f"  Feature value:                  {features['expected_net_position'].iloc[100]:>12,.2f}")
print(f"  Computed sales-purchases:       {enp_check:>12,.2f}")
print(f"  Match: {np.isclose(features['expected_net_position'].iloc[100], enp_check)}")

print(f"\nCurrent feature count: {features.shape[1]} columns")

In [ ]:
print(len(features.columns), "columns total")
for c in features.columns:
    print(f"  {c}")

In [ ]:
# Payment-cycle feature visualisation
# ------------------------------------
# Shows how the engineered `expected_net_position` — built purely from
# past invoices in the 15-90 day payment window — tracks alongside the
# actual weekly net cashflow. Evidence that the feature captures the
# information it was designed to capture.

import matplotlib.dates as mdates
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Top panel — actual net cashflow (target)
axes[0].plot(features.index, features['net_cashflow'],
             color='darkgreen', linewidth=1.1)
axes[0].axhline(0, color='black', linestyle='--', linewidth=0.8, alpha=0.6)
axes[0].fill_between(features.index, features['net_cashflow'], 0,
                     where=(features['net_cashflow'] >= 0),
                     color='green', alpha=0.2)
axes[0].fill_between(features.index, features['net_cashflow'], 0,
                     where=(features['net_cashflow'] < 0),
                     color='red', alpha=0.2)
axes[0].set_ylabel('Actual weekly net cashflow (INR)')
axes[0].set_title('Target variable: actual weekly net cashflow')
axes[0].grid(True, alpha=0.3)

# Bottom panel — expected_net_position (engineered feature)
axes[1].plot(features.index, features['expected_net_position'],
             color='navy', linewidth=1.1)
axes[1].axhline(0, color='black', linestyle='--', linewidth=0.8, alpha=0.6)
axes[1].fill_between(features.index, features['expected_net_position'], 0,
                     where=(features['expected_net_position'] >= 0),
                     color='blue', alpha=0.2)
axes[1].fill_between(features.index, features['expected_net_position'], 0,
                     where=(features['expected_net_position'] < 0),
                     color='orange', alpha=0.2)

In [ ]:
# Explicit correlation + a few diagnostics
valid = features[['net_cashflow', 'expected_net_position']].dropna()

print(f"Rows used: {len(valid)}")
print(f"\nPearson correlation:  {valid['net_cashflow'].corr(valid['expected_net_position']):.3f}")
print(f"Spearman correlation: {valid['net_cashflow'].corr(valid['expected_net_position'], method='spearman'):.3f}")

print(f"\nScale check:")
print(f"  net_cashflow mean:              {valid['net_cashflow'].mean():>14,.2f}")
print(f"  net_cashflow std:               {valid['net_cashflow'].std():>14,.2f}")
print(f"  expected_net_position mean:     {valid['expected_net_position'].mean():>14,.2f}")
print(f"  expected_net_position std:      {valid['expected_net_position'].std():>14,.2f}")

In [ ]:
# A tighter payment-cycle feature — single point-lag at 6 weeks (~45 days)
features['sales_expected_45d'] = features['sales'].shift(6)
features['purchases_expected_45d'] = features['purchases'].shift(6)
features['net_expected_45d'] = (features['sales_expected_45d'] 
                                - features['purchases_expected_45d'])

# Check correlation of this tighter feature vs the target
valid = features[['net_cashflow', 'net_expected_45d']].dropna()

print(f"Tighter payment-cycle feature (single 6-week / ~45-day lag):")
print(f"  Rows used: {len(valid)}")
print(f"  Pearson correlation:  {valid['net_cashflow'].corr(valid['net_expected_45d']):.3f}")
print(f"  Spearman correlation: {valid['net_cashflow'].corr(valid['net_expected_45d'], method='spearman'):.3f}")

# Feature count
print(f"\nCurrent feature count: {features.shape[1]} columns")

In [ ]:
print(f"features shape: {features.shape}")
print(f"features columns: {list(features.columns)}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings, os

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

# --- Load cleaned invoice data ---
sales_clean    = pd.read_parquet("../data/processed/sales_clean.parquet")
purchase_clean = pd.read_parquet("../data/processed/purchase_clean.parquet")

# --- Load the accountant's completed terms file ---
TERMS_FILE = "../data/private/customer_terms_template.xlsx"
buyer_terms    = pd.read_excel(TERMS_FILE, sheet_name='Customers (Sales)')
supplier_terms = pd.read_excel(TERMS_FILE, sheet_name='Suppliers (Purchases)')

print(f"Sales invoices:     {len(sales_clean):,}")
print(f"Purchase invoices:  {len(purchase_clean):,}")
print(f"Buyers with terms:      {buyer_terms['Payment Terms (days)'].notna().sum()} / {len(buyer_terms)}")
print(f"Suppliers with terms:   {supplier_terms['Payment Terms (days)'].notna().sum()} / {len(supplier_terms)}")

# Display terms distribution WITHOUT real names — for privacy in notebook outputs
print("\nBuyer terms — payment terms distribution:")
print(buyer_terms['Payment Terms (days)'].value_counts().to_string())

print("\nSupplier terms — payment terms distribution:")
print(supplier_terms['Payment Terms (days)'].value_counts().to_string())

print("\nBuyer terms — first 3 rows (identifying columns hidden):")
display(buyer_terms.head(3)[['Number of Invoices', 'First Invoice', 
                              'Last Invoice', 'Payment Terms (days)']])

print("\nSupplier terms — first 3 rows (identifying columns hidden):")
display(supplier_terms.head(3)[['Number of Invoices', 'First Invoice', 
                                 'Last Invoice', 'Payment Terms (days)']])

In [ ]:
# Clean whitespace in terms columns
# ---------------------------------
# The accountant occasionally added trailing spaces on some entries.
# "7" and "7 " are different strings to Python — must be stripped to avoid
# silent misclassification during parsing.

def clean_terms(series):
    """Strip whitespace and standardise the payment terms strings."""
    return series.astype(str).str.strip()

buyer_terms['Payment Terms (days)']    = clean_terms(buyer_terms['Payment Terms (days)'])
supplier_terms['Payment Terms (days)'] = clean_terms(supplier_terms['Payment Terms (days)'])

# Cleaned distributions (consolidated after whitespace strip)
print("=" * 60)
print("BUYER terms — cleaned distribution:")
print("=" * 60)
for term, count in buyer_terms['Payment Terms (days)'].value_counts().sort_index().items():
    print(f"  '{term}'  →  {count} buyers")

print("\n" + "=" * 60)
print("SUPPLIER terms — cleaned distribution:")
print("=" * 60)
for term, count in supplier_terms['Payment Terms (days)'].value_counts().sort_index().items():
    print(f"  '{term}'  →  {count} suppliers")

print(f"\nAll unique supplier term strings: {sorted(supplier_terms['Payment Terms (days)'].unique())}")

In [ ]:
# Map real GSTINs in the accountant's terms file to pseudonyms
# -------------------------------------------------------------
# The accountant filled in real GSTINs. Our invoice data uses pseudonymised
# GSTINs (Day 4 anonymisation). We convert here, then immediately drop the
# real GSTIN column so real identifiers never leave this cell.

import json

with open("../data/private/gstin_mapping.json", "r") as f:
    gstin_map_forward = json.load(f)   # real_GSTIN → pseudonym

# Convert real GSTINs to pseudonyms in both terms files
buyer_terms['GSTIN_pseudo']    = buyer_terms['GSTIN'].map(gstin_map_forward)
supplier_terms['GSTIN_pseudo'] = supplier_terms['GSTIN'].map(gstin_map_forward)

buyer_mapped    = buyer_terms['GSTIN_pseudo'].notna().sum()
supplier_mapped = supplier_terms['GSTIN_pseudo'].notna().sum()

print(f"BUYER terms — GSTIN mapping:")
print(f"  Successfully mapped: {buyer_mapped} / {len(buyer_terms)}  "
      f"({100 * buyer_mapped / len(buyer_terms):.1f}%)")
print(f"  Failed to map:       {len(buyer_terms) - buyer_mapped}")

print(f"\nSUPPLIER terms — GSTIN mapping:")
print(f"  Successfully mapped: {supplier_mapped} / {len(supplier_terms)}  "
      f"({100 * supplier_mapped / len(supplier_terms):.1f}%)")
print(f"  Failed to map:       {len(supplier_terms) - supplier_mapped}")

# Drop the real GSTIN column now — from here on, only pseudonyms in memory
buyer_terms    = buyer_terms.drop(columns=['GSTIN']).rename(columns={'GSTIN_pseudo': 'GSTIN'})
supplier_terms = supplier_terms.drop(columns=['GSTIN']).rename(columns={'GSTIN_pseudo': 'GSTIN'})

print("\nReal GSTINs dropped from working DataFrames. Only pseudonyms retained.")

In [ ]:
# Diagnose: what's in the GSTIN columns now, and what's in the mapping?

print("=" * 60)
print("What GSTIN values look like in terms files RIGHT NOW:")
print("=" * 60)

print("\nSample 5 GSTIN values from buyer_terms (after Cell 3 ran):")
print(buyer_terms['GSTIN'].head(5).to_list())

print("\nSample 5 GSTIN values from supplier_terms:")
print(supplier_terms['GSTIN'].head(5).to_list())

print("\n" + "=" * 60)
print("What's in the gstin_mapping.json file:")
print("=" * 60)

with open("../data/private/gstin_mapping.json", "r") as f:
    gm = json.load(f)

print(f"\nTotal mappings in file: {len(gm)}")
print(f"\nFirst 3 keys (real GSTINs):")
for k in list(gm.keys())[:3]:
    print(f"  '{k}'")
print(f"\nFirst 3 values (pseudonyms):")
for v in list(gm.values())[:3]:
    print(f"  '{v}'")

print("\n" + "=" * 60)
print("Direction check:")
print("=" * 60)
print(f"buyer_terms GSTIN sample looks like: real (29XXX...) or pseudo (GSTIN_XXX)?")
print(f"Mapping keys look like: real or pseudo?")

In [ ]:
# Re-read the terms file fresh from Excel to see what it actually contains
TERMS_FILE = "../data/private/customer_terms_template.xlsx"

fresh_buyer    = pd.read_excel(TERMS_FILE, sheet_name='Customers (Sales)')
fresh_supplier = pd.read_excel(TERMS_FILE, sheet_name='Suppliers (Purchases)')

print("Fresh buyer_terms columns:", list(fresh_buyer.columns))
print("Fresh supplier_terms columns:", list(fresh_supplier.columns))

print(f"\nBuyer GSTIN column null count: {fresh_buyer['GSTIN'].isnull().sum() if 'GSTIN' in fresh_buyer.columns else 'GSTIN COLUMN NOT PRESENT'}")
print(f"\nBuyer GSTIN sample values (first 5):")
if 'GSTIN' in fresh_buyer.columns:
    print(fresh_buyer['GSTIN'].head(5).to_list())

print(f"\nSupplier GSTIN column null count: {fresh_supplier['GSTIN'].isnull().sum() if 'GSTIN' in fresh_supplier.columns else 'GSTIN COLUMN NOT PRESENT'}")
print(f"\nSupplier GSTIN sample values (first 5):")
if 'GSTIN' in fresh_supplier.columns:
    print(fresh_supplier['GSTIN'].head(5).to_list())

In [ ]:
# Reload terms fresh from Excel and remap GSTINs cleanly
# ------------------------------------------------------
# The earlier attempt corrupted the in-memory DataFrames by mapping against
# a null column. Re-read from disk to get real GSTINs back, then map cleanly.

TERMS_FILE = "../data/private/customer_terms_template.xlsx"

buyer_terms    = pd.read_excel(TERMS_FILE, sheet_name='Customers (Sales)')
supplier_terms = pd.read_excel(TERMS_FILE, sheet_name='Suppliers (Purchases)')

# Re-apply whitespace cleaning (from Cell 2 logic) since we reloaded
buyer_terms['Payment Terms (days)']    = buyer_terms['Payment Terms (days)'].astype(str).str.strip()
supplier_terms['Payment Terms (days)'] = supplier_terms['Payment Terms (days)'].astype(str).str.strip()

# Load the GSTIN mapping (should already be in memory from earlier, but load defensively)
with open("../data/private/gstin_mapping.json", "r") as f:
    gstin_map_forward = json.load(f)   # real_GSTIN → pseudonym

# Map real GSTINs to pseudonyms
buyer_terms['GSTIN_pseudo']    = buyer_terms['GSTIN'].map(gstin_map_forward)
supplier_terms['GSTIN_pseudo'] = supplier_terms['GSTIN'].map(gstin_map_forward)

buyer_mapped    = buyer_terms['GSTIN_pseudo'].notna().sum()
supplier_mapped = supplier_terms['GSTIN_pseudo'].notna().sum()

print(f"BUYER terms — GSTIN mapping:")
print(f"  Successfully mapped: {buyer_mapped} / {len(buyer_terms)}  "
      f"({100 * buyer_mapped / len(buyer_terms):.1f}%)")

print(f"\nSUPPLIER terms — GSTIN mapping:")
print(f"  Successfully mapped: {supplier_mapped} / {len(supplier_terms)}  "
      f"({100 * supplier_mapped / len(supplier_terms):.1f}%)")

# Drop the real GSTIN column, keep only the pseudonym
buyer_terms    = buyer_terms.drop(columns=['GSTIN']).rename(columns={'GSTIN_pseudo': 'GSTIN'})
supplier_terms = supplier_terms.drop(columns=['GSTIN']).rename(columns={'GSTIN_pseudo': 'GSTIN'})

# Sanity check
print(f"\nAfter mapping, sample buyer GSTINs (should be pseudonyms):")
print(buyer_terms['GSTIN'].head(3).to_list())

In [ ]:
# Deduplicate the terms files by GSTIN
# ------------------------------------
# Tally allows the same real company to be entered under slightly different
# name spellings, so a single GSTIN can appear in multiple rows. Terms are
# consistent across duplicates (verified this morning), so we keep the row
# with the most invoices for each GSTIN as the canonical entry.

def dedupe_terms(df):
    """Keep one row per GSTIN — the one with the most invoices."""
    return (
        df
        .sort_values('Number of Invoices', ascending=False)
        .drop_duplicates(subset='GSTIN', keep='first')
        .reset_index(drop=True)
    )

# Consistency check first — flag any GSTIN with differing terms across rows
def check_term_consistency(df, name):
    inconsistent = df.groupby('GSTIN')['Payment Terms (days)'].nunique()
    inconsistent = inconsistent[inconsistent > 1]
    if len(inconsistent) > 0:
        print(f"⚠ {name}: {len(inconsistent)} GSTINs have DIFFERENT terms across duplicate rows")
    else:
        print(f"✓ {name}: all GSTIN duplicates have consistent terms (safe to dedup)")

# Drop the 18 unmapped supplier rows before dedup — they carry no GSTIN so
# they'd all collapse into a single NaN row which we don't want
supplier_terms_clean = supplier_terms.dropna(subset=['GSTIN']).copy()

check_term_consistency(buyer_terms,          "Buyer terms")
check_term_consistency(supplier_terms_clean, "Supplier terms")

buyer_terms_dedup    = dedupe_terms(buyer_terms)
supplier_terms_dedup = dedupe_terms(supplier_terms_clean)

print(f"\nBuyer terms:    {len(buyer_terms)} rows → {len(buyer_terms_dedup)} unique GSTINs")
print(f"Supplier terms: {len(supplier_terms_clean)} rows → {len(supplier_terms_dedup)} unique GSTINs")

In [ ]:
# Payment terms parser and projection functions
# ---------------------------------------------
# parse_terms:              "30" -> (30,30);  "15-30" -> (15,30)
# project_to_payment_day:   apply size-conditioning + 2-day banking delay
# day_to_week_offset:       convert day-offset into week-offset

import re

SIZE_THRESHOLD = 22_500   # fixed threshold — will be replaced with per-customer later
BANKING_DELAY  = 2        # days added to every non-immediate term

def parse_terms(term_str):
    """Parse '30' -> (30,30); '15-30' -> (15,30). Raises ValueError on unknown format."""
    term_str = term_str.strip()
    match_range  = re.fullmatch(r'(\d+)\s*-\s*(\d+)', term_str)
    match_single = re.fullmatch(r'(\d+)', term_str)
    
    if match_range:
        return int(match_range.group(1)), int(match_range.group(2))
    elif match_single:
        n = int(match_single.group(1))
        return n, n
    else:
        raise ValueError(f"Unrecognised term format: '{term_str}'")


def project_to_payment_day(term_str, invoice_amount):
    """Return day-offset from invoice date when payment is expected."""
    start, end = parse_terms(term_str)
    if start == end:
        return start + BANKING_DELAY
    else:
        if invoice_amount >= SIZE_THRESHOLD:
            return end + BANKING_DELAY
        else:
            return start + BANKING_DELAY


def day_to_week_offset(day):
    """Convert day-offset to week-offset. Day 0 -> week 0; days 1-7 -> week 1; etc."""
    if day == 0:
        return 0
    return (day - 1) // 7 + 1


# Verify parser handles every unique term string in the data
all_terms = set(buyer_terms_dedup['Payment Terms (days)'].unique()) | \
            set(supplier_terms_dedup['Payment Terms (days)'].unique())

print("Parser verification — every unique term string in the data:")
print(f"{'Term':<10}{'Start':>7}{'End':>7}{'Small→wk':>12}{'Large→wk':>12}")
print("-" * 48)
for t in sorted(all_terms, key=lambda x: (len(x), x)):
    start, end = parse_terms(t)
    small_day = project_to_payment_day(t, 10_000)
    large_day = project_to_payment_day(t, 50_000)
    small_wk  = day_to_week_offset(small_day)
    large_wk  = day_to_week_offset(large_day)
    print(f"{t:<10}{start:>7}{end:>7}{small_wk:>12}{large_wk:>12}")

print(f"\nAll {len(all_terms)} unique term strings parsed successfully.")

In [ ]:
# Merge terms onto invoices, compute default, and fill unmatched
# --------------------------------------------------------------
# Left-join by pseudonymised GSTIN. Unmatched invoices (from suppliers
# missing from the terms file) get a weighted-average default term.

# --- Compute the weighted-average default from mapped suppliers ---
def term_midpoint_days(t):
    start, end = parse_terms(t)
    return (start + end) / 2

supplier_terms_dedup['_midpoint'] = (
    supplier_terms_dedup['Payment Terms (days)'].apply(term_midpoint_days)
)

weighted_avg_days = (
    (supplier_terms_dedup['_midpoint'] * supplier_terms_dedup['Number of Invoices']).sum()
    / supplier_terms_dedup['Number of Invoices'].sum()
)
DEFAULT_TERM = f"{int(round(weighted_avg_days))}"

print(f"Weighted-average supplier term (from mapped suppliers): {weighted_avg_days:.1f} days")
print(f"Default term for unmatched invoices: '{DEFAULT_TERM}'")

# --- Merge terms onto sales and purchase invoices ---
n_sales_before    = len(sales_clean)
n_purchase_before = len(purchase_clean)

sales_with_terms = sales_clean.merge(
    buyer_terms_dedup[['GSTIN', 'Payment Terms (days)']],
    left_on='GSTIN/UIN',
    right_on='GSTIN',
    how='left'
).drop(columns=['GSTIN'])

purchase_with_terms = purchase_clean.merge(
    supplier_terms_dedup[['GSTIN', 'Payment Terms (days)']],
    left_on='GSTIN/UIN',
    right_on='GSTIN',
    how='left'
).drop(columns=['GSTIN'])

# Row-count verification — the merge should NOT inflate row counts
print(f"\nMerge row-count check:")
print(f"  Sales:     {n_sales_before} → {len(sales_with_terms)}  "
      f"({'OK' if n_sales_before == len(sales_with_terms) else 'INFLATED'})")
print(f"  Purchases: {n_purchase_before} → {len(purchase_with_terms)}  "
      f"({'OK' if n_purchase_before == len(purchase_with_terms) else 'INFLATED'})")

# Match rates before default fill
sales_matched    = sales_with_terms['Payment Terms (days)'].notna().sum()
purchase_matched = purchase_with_terms['Payment Terms (days)'].notna().sum()
print(f"\nSales matched to real term:     {sales_matched}/{len(sales_with_terms)}  "
      f"({100*sales_matched/len(sales_with_terms):.1f}%)")
print(f"Purchases matched to real term: {purchase_matched}/{len(purchase_with_terms)}  "
      f"({100*purchase_matched/len(purchase_with_terms):.1f}%)")

# Apply default term to any unmatched rows
sales_with_terms['Payment Terms (days)']    = sales_with_terms['Payment Terms (days)'].fillna(DEFAULT_TERM)
purchase_with_terms['Payment Terms (days)'] = purchase_with_terms['Payment Terms (days)'].fillna(DEFAULT_TERM)

print(f"\nAfter default fill:")
print(f"  Sales with term:     {sales_with_terms['Payment Terms (days)'].notna().sum()}/{len(sales_with_terms)}")
print(f"  Purchases with term: {purchase_with_terms['Payment Terms (days)'].notna().sum()}/{len(purchase_with_terms)}")

In [ ]:
# Project each invoice to its expected payment week + build v1 target
# -------------------------------------------------------------------
# Uses the fixed ₹22,500 size-conditioning threshold. This is the
# "first version" of the payment-projected target; per-customer
# thresholds come in the next iteration.

def project_invoice(row):
    """Project one invoice to its expected payment date."""
    day_offset  = project_to_payment_day(row['Payment Terms (days)'], row['Gross Total'])
    week_offset = day_to_week_offset(day_offset)
    return row['Date'] + pd.Timedelta(weeks=week_offset)

print("Projecting sales invoices...")
sales_with_terms['projected_payment_date'] = sales_with_terms.apply(project_invoice, axis=1)

print("Projecting purchase invoices...")
purchase_with_terms['projected_payment_date'] = purchase_with_terms.apply(project_invoice, axis=1)

# Aggregate to weekly totals on the projected date
sales_weekly_projected = (
    sales_with_terms
    .set_index('projected_payment_date')['Gross Total']
    .resample('W').sum()
    .rename('sales')
)

purchase_weekly_projected = (
    purchase_with_terms
    .set_index('projected_payment_date')['Gross Total']
    .resample('W').sum()
    .rename('purchases')
)

# Join into a single DataFrame
cashflow_projected = pd.concat([sales_weekly_projected, purchase_weekly_projected], axis=1).fillna(0)
cashflow_projected['net_cashflow'] = cashflow_projected['sales'] - cashflow_projected['purchases']

print(f"\nPayment-projected weekly cashflow (v1, fixed threshold):")
print(f"  Weeks in series:            {len(cashflow_projected):,}")
print(f"  Date range:                 {cashflow_projected.index.min().date()} → "
      f"{cashflow_projected.index.max().date()}")
print(f"  Weeks with negative net:    {(cashflow_projected['net_cashflow'] < 0).sum()} "
      f"({100 * (cashflow_projected['net_cashflow'] < 0).mean():.1f}%)")

print(f"\nDescriptive stats — payment-projected weekly net cashflow (INR):")
print(cashflow_projected['net_cashflow'].describe().apply('{:,.2f}'.format))

In [ ]:
# Correlation diagnostic — payment-cycle feature vs v1 payment-projected target
# ------------------------------------------------------------------------------
# Rebuild the two key payment-cycle features on the new target's index,
# then compute correlations and produce the 3-panel diagnostic figure.

from scipy import stats

# Rebuild the payment-cycle features against the new target's index
temp = cashflow_projected.copy()
temp['sales_paywindow_full']     = temp['sales'].shift(1).rolling(12, min_periods=12).sum()
temp['purchases_paywindow_full'] = temp['purchases'].shift(1).rolling(12, min_periods=12).sum()
temp['expected_net_position']    = temp['sales_paywindow_full'] - temp['purchases_paywindow_full']
temp['net_expected_45d']         = temp['sales'].shift(6) - temp['purchases'].shift(6)

# --- Correlation numbers ---
valid = temp[['net_cashflow', 'expected_net_position']].dropna()
pearson_r  = valid['net_cashflow'].corr(valid['expected_net_position'])
spearman_r = valid['net_cashflow'].corr(valid['expected_net_position'], method='spearman')

tight_valid = temp[['net_cashflow', 'net_expected_45d']].dropna()
pearson_tight  = tight_valid['net_cashflow'].corr(tight_valid['net_expected_45d'])
spearman_tight = tight_valid['net_cashflow'].corr(tight_valid['net_expected_45d'], method='spearman')

print("=" * 60)
print("PAYMENT-CYCLE FEATURE vs PAYMENT-PROJECTED TARGET (v1)")
print("=" * 60)
print(f"\nexpected_net_position (12-week aggregate):")
print(f"  Pearson:  {pearson_r:.3f}")
print(f"  Spearman: {spearman_r:.3f}")
print(f"\nnet_expected_45d (single 6-week lag):")
print(f"  Pearson:  {pearson_tight:.3f}")
print(f"  Spearman: {spearman_tight:.3f}")

# --- Prepare data for visualisation ---
plot_data = temp[['net_cashflow', 'expected_net_position']].dropna().copy()
plot_data['target_z']  = ((plot_data['net_cashflow'] - plot_data['net_cashflow'].mean())
                          / plot_data['net_cashflow'].std())
plot_data['feature_z'] = ((plot_data['expected_net_position'] - plot_data['expected_net_position'].mean())
                          / plot_data['expected_net_position'].std())
rolling_corr = plot_data['net_cashflow'].rolling(12).corr(plot_data['expected_net_position'])

# --- 3-panel diagnostic figure ---
fig, axes = plt.subplots(3, 1, figsize=(14, 11))

# Panel 1: time-series overlay
axes[0].plot(plot_data.index, plot_data['target_z'],
             color='darkgreen', linewidth=1.2, alpha=0.8,
             label='Payment-projected net cashflow (target)')
axes[0].plot(plot_data.index, plot_data['feature_z'],
             color='navy', linewidth=1.2, alpha=0.8,
             label='expected_net_position (feature)')
axes[0].axhline(0, color='black', linestyle='--', linewidth=0.6, alpha=0.4)
axes[0].set_ylabel('Standardised value (z-score)')
axes[0].set_title('Time-series overlay: feature vs target (standardised for scale)')
axes[0].legend(loc='upper left', fontsize=9)
axes[0].grid(True, alpha=0.3)

# Panel 2: scatter
axes[1].scatter(plot_data['expected_net_position'], plot_data['net_cashflow'],
                alpha=0.5, s=25, color='steelblue', edgecolor='white', linewidth=0.5)
slope, intercept, _, _, _ = stats.linregress(plot_data['expected_net_position'],
                                              plot_data['net_cashflow'])
x_range = np.array([plot_data['expected_net_position'].min(),
                    plot_data['expected_net_position'].max()])
axes[1].plot(x_range, slope * x_range + intercept, color='red', linewidth=1.2, alpha=0.7,
             label=f'Best-fit line (slope = {slope:.3f})')
axes[1].axhline(0, color='black', linestyle='--', linewidth=0.6, alpha=0.4)
axes[1].axvline(0, color='black', linestyle='--', linewidth=0.6, alpha=0.4)
axes[1].set_xlabel('expected_net_position (feature)')
axes[1].set_ylabel('Payment-projected net cashflow (target)')
axes[1].set_title(f'Scatter: feature vs target  |  '
                  f'Pearson r = {pearson_r:.3f}, Spearman = {spearman_r:.3f}')
axes[1].legend(loc='upper left', fontsize=9)
axes[1].grid(True, alpha=0.3)

# Panel 3: rolling correlation
axes[2].plot(rolling_corr.index, rolling_corr, color='purple', linewidth=1.1)
axes[2].axhline(0, color='black', linestyle='--', linewidth=0.6, alpha=0.4)
axes[2].axhline(pearson_r, color='red', linestyle=':', linewidth=1, alpha=0.6,
                label=f'Overall Pearson = {pearson_r:.3f}')
axes[2].fill_between(rolling_corr.index, 0, rolling_corr,
                     where=(rolling_corr >= 0), color='green', alpha=0.15,
                     label='Positive correlation window')
axes[2].fill_between(rolling_corr.index, 0, rolling_corr,
                     where=(rolling_corr < 0), color='red', alpha=0.15,
                     label='Negative correlation window')
axes[2].set_ylabel('Rolling correlation coefficient')
axes[2].set_title('12-week rolling correlation between feature and target')
axes[2].legend(loc='upper left', fontsize=9)
axes[2].grid(True, alpha=0.3)
axes[2].xaxis.set_major_locator(mdates.MonthLocator(interval=3))
axes[2].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.setp(axes[2].xaxis.get_majorticklabels(), rotation=45, ha='right')

fig.suptitle('Payment-cycle feature vs payment-projected target — diagnostic',
             fontsize=13, y=1.00)
plt.tight_layout()

os.makedirs("../reports/figures", exist_ok=True)
plt.savefig("../reports/figures/paycycle_correlation_diagnostic.png",
            bbox_inches='tight', dpi=150)
plt.show()

print("\nDiagnostic figure saved to reports/figures/paycycle_correlation_diagnostic.png")

In [ ]:
# Per-customer size thresholds — replace fixed ₹22,500 with per-customer p75
# --------------------------------------------------------------------------
# The SME accountant clarified that "large" invoice size is customer-relative:
# a ₹15,000 invoice may be significant for a small customer and trivial for a
# large one. Rebuild size-conditioning using each customer's own 75th-percentile
# invoice value with the SME.

# Compute per-buyer 75th percentile of Gross Total (pseudonymised GSTIN key)
per_buyer_threshold = (
    sales_with_terms
    .groupby('GSTIN/UIN')['Gross Total']
    .quantile(0.75)
    .rename('customer_threshold')
)

print(f"Per-customer thresholds computed for {len(per_buyer_threshold)} unique buyers.")
print(f"\nDistribution of the thresholds themselves (INR):")
print(per_buyer_threshold.describe().apply('{:,.0f}'.format))

# Attach threshold to each sales invoice
sales_with_terms = sales_with_terms.merge(
    per_buyer_threshold.reset_index(),
    on='GSTIN/UIN',
    how='left'
)

# Comparison: old rule vs new rule
old_large = (sales_with_terms['Gross Total'] >= 22_500).sum()
new_large = (sales_with_terms['Gross Total'] >= sales_with_terms['customer_threshold']).sum()

print(f"\nComparison of 'large' classification:")
print(f"  Old rule (fixed ₹22,500):     {old_large} invoices flagged as large "
      f"({100*old_large/len(sales_with_terms):.1f}%)")
print(f"  New rule (per-customer p75):  {new_large} invoices flagged as large "
      f"({100*new_large/len(sales_with_terms):.1f}%)")


def project_invoice_customer_relative(row):
    """
    Same projection logic, but uses per-customer threshold instead of fixed ₹22,500.
    Falls back to fixed threshold if per-customer value is missing.
    """
    start, end = parse_terms(row['Payment Terms (days)'])
    
    if start == end:
        day_offset = start + BANKING_DELAY
    else:
        threshold = row['customer_threshold'] if pd.notna(row['customer_threshold']) else 22_500
        if row['Gross Total'] >= threshold:
            day_offset = end + BANKING_DELAY
        else:
            day_offset = start + BANKING_DELAY
    
    week_offset = day_to_week_offset(day_offset)
    return row['Date'] + pd.Timedelta(weeks=week_offset)


print("\nRe-projecting sales invoices with per-customer thresholds...")
sales_with_terms['projected_payment_date_v2'] = sales_with_terms.apply(
    project_invoice_customer_relative, axis=1
)

# Purchases stay on the fixed-threshold projection (no per-supplier work today)
purchase_with_terms['projected_payment_date_v2'] = purchase_with_terms['projected_payment_date']

# How many invoices shifted?
diff_days = (sales_with_terms['projected_payment_date_v2']
             - sales_with_terms['projected_payment_date']).dt.days
n_shifted = (diff_days != 0).sum()

print(f"\nSales invoices with different projected date under new rule: "
      f"{n_shifted} ({100*n_shifted/len(sales_with_terms):.1f}%)")
print(f"Shift distribution (days):")
print(diff_days[diff_days != 0].describe().apply('{:,.1f}'.format))

# Build the v2 payment-projected target
sales_weekly_v2 = (
    sales_with_terms
    .set_index('projected_payment_date_v2')['Gross Total']
    .resample('W').sum()
    .rename('sales')
)
purchase_weekly_v2 = (
    purchase_with_terms
    .set_index('projected_payment_date_v2')['Gross Total']
    .resample('W').sum()
    .rename('purchases')
)

cashflow_projected_v2 = pd.concat([sales_weekly_v2, purchase_weekly_v2], axis=1).fillna(0)
cashflow_projected_v2['net_cashflow'] = (cashflow_projected_v2['sales']
                                          - cashflow_projected_v2['purchases'])

print(f"\nPayment-projected weekly cashflow (v2, per-customer thresholds):")
print(f"  Weeks in series:           {len(cashflow_projected_v2)}")
print(f"  Weeks with negative net:   {(cashflow_projected_v2['net_cashflow'] < 0).sum()} "
      f"({100 * (cashflow_projected_v2['net_cashflow'] < 0).mean():.1f}%)")
print(f"  Median net cashflow:       {cashflow_projected_v2['net_cashflow'].median():,.2f}")

In [ ]:
# Correlation diagnostic — payment-cycle features vs v2 target
# ------------------------------------------------------------
# If the per-customer threshold rebuild meaningfully improves correlation,
# the refinement was worth it. If not, we've confirmed the structural nature
# of the near-zero correlation.

temp_v2 = cashflow_projected_v2.copy()
temp_v2['sales_paywindow_full']     = temp_v2['sales'].shift(1).rolling(12, min_periods=12).sum()
temp_v2['purchases_paywindow_full'] = temp_v2['purchases'].shift(1).rolling(12, min_periods=12).sum()
temp_v2['expected_net_position']    = temp_v2['sales_paywindow_full'] - temp_v2['purchases_paywindow_full']
temp_v2['net_expected_45d']         = temp_v2['sales'].shift(6) - temp_v2['purchases'].shift(6)

valid_v2 = temp_v2[['net_cashflow', 'expected_net_position']].dropna()
tight_v2 = temp_v2[['net_cashflow', 'net_expected_45d']].dropna()

print("=" * 60)
print("CORRELATION AGAINST v2 TARGET (per-customer thresholds)")
print("=" * 60)
print(f"\nexpected_net_position feature:")
print(f"  Pearson:  {valid_v2['net_cashflow'].corr(valid_v2['expected_net_position']):.3f}")
print(f"  Spearman: {valid_v2['net_cashflow'].corr(valid_v2['expected_net_position'], method='spearman'):.3f}")

print(f"\nnet_expected_45d feature:")
print(f"  Pearson:  {tight_v2['net_cashflow'].corr(tight_v2['net_expected_45d']):.3f}")
print(f"  Spearman: {tight_v2['net_cashflow'].corr(tight_v2['net_expected_45d'], method='spearman'):.3f}")

print("\n" + "-" * 60)
print("For comparison — v1 target (fixed threshold):")
print("  expected_net_position — Pearson: -0.049, Spearman:  0.047")
print("  net_expected_45d      — Pearson: -0.095, Spearman: -0.127")
print("-" * 60)
print("\nInterpretation: correlations essentially unchanged, confirming the")
print("issue is structural (aggregate features cannot linearly capture")
print("single-week noise) rather than a threshold-tuning problem.")

In [ ]:
# Save v2 target and per-invoice projections
# ------------------------------------------
# The v2 payment-projected target becomes the primary cash-basis series
# for Phase 3 modelling, alongside the accrual target from Phase 2 core.

# The Voucher No. columns have mixed types (Tally exports produce a mix
# of ints, floats, and strings). Force to string for Parquet compatibility.
for df in (sales_with_terms, purchase_with_terms):
    if 'Voucher No.' in df.columns:
        df['Voucher No.'] = df['Voucher No.'].astype(str)

# Save the v2 target as the primary payment-projected target
cashflow_projected_v2.to_parquet("../data/processed/weekly_cashflow_projected.parquet")

# Save the per-invoice projections (both v1 and v2 dates included)
sales_with_terms.to_parquet("../data/processed/sales_with_projections.parquet")
purchase_with_terms.to_parquet("../data/processed/purchase_with_projections.parquet")

print("Saved:")
print("  ../data/processed/weekly_cashflow_projected.parquet    (v2 target, 163 weeks)")
print("  ../data/processed/sales_with_projections.parquet       (per-invoice projections)")
print("  ../data/processed/purchase_with_projections.parquet    (per-invoice projections)")
print()
print("Existing on disk (from Phase 2 core):")
print("  ../data/processed/weekly_cashflow.parquet              (accrual target, 158 weeks)")

In [ ]:
# Phase 2 closing visualisation — the two targets and their characteristics
# --------------------------------------------------------------------------
# 4-panel figure summarising Phase 2 output for the dissertation:
#   - Panel 1: accrual target time series
#   - Panel 2: payment-projected target time series
#   - Panel 3: distribution overlay
#   - Panel 4: cumulative comparison
#   - Bottom: summary statistics table

cf_accrual   = pd.read_parquet("../data/processed/weekly_cashflow.parquet")
cf_projected = pd.read_parquet("../data/processed/weekly_cashflow_projected.parquet")

def summarise(cf, label):
    net = cf['net_cashflow']
    return {
        'label':       label,
        'weeks':       len(cf),
        'mean':        net.mean(),
        'median':      net.median(),
        'std':         net.std(),
        'min':         net.min(),
        'max':         net.max(),
        'deficit_pct': 100 * (net < 0).mean(),
    }

s_accrual   = summarise(cf_accrual,   'Accrual (invoice-issue date)')
s_projected = summarise(cf_projected, 'Payment-projected (per-customer terms)')

fig = plt.figure(figsize=(16, 11))
gs  = fig.add_gridspec(3, 2, height_ratios=[1.1, 1.1, 0.9], hspace=0.45, wspace=0.25)

# Panel 1 — Accrual target time series
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(cf_accrual.index, cf_accrual['net_cashflow'], color='darkgreen', linewidth=1.0)
ax1.axhline(0, color='black', linestyle='--', linewidth=0.6, alpha=0.5)
ax1.fill_between(cf_accrual.index, cf_accrual['net_cashflow'], 0,
                 where=(cf_accrual['net_cashflow'] >= 0), color='green', alpha=0.2)
ax1.fill_between(cf_accrual.index, cf_accrual['net_cashflow'], 0,
                 where=(cf_accrual['net_cashflow'] < 0), color='red', alpha=0.2)
ax1.set_ylabel('Weekly net cashflow (INR)')
ax1.set_title(f"Accrual target ({s_accrual['weeks']} weeks) — "
              f"deficit weeks: {s_accrual['deficit_pct']:.1f}%", fontsize=11)
ax1.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45, ha='right')
ax1.grid(True, alpha=0.3)

# Panel 2 — Payment-projected target time series
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(cf_projected.index, cf_projected['net_cashflow'], color='navy', linewidth=1.0)
ax2.axhline(0, color='black', linestyle='--', linewidth=0.6, alpha=0.5)
ax2.fill_between(cf_projected.index, cf_projected['net_cashflow'], 0,
                 where=(cf_projected['net_cashflow'] >= 0), color='blue', alpha=0.2)
ax2.fill_between(cf_projected.index, cf_projected['net_cashflow'], 0,
                 where=(cf_projected['net_cashflow'] < 0), color='red', alpha=0.2)
ax2.set_ylabel('Weekly net cashflow (INR)')
ax2.set_title(f"Payment-projected target ({s_projected['weeks']} weeks) — "
              f"deficit weeks: {s_projected['deficit_pct']:.1f}%", fontsize=11)
ax2.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45, ha='right')
ax2.grid(True, alpha=0.3)

# Panel 3 — Distribution overlay
ax3 = fig.add_subplot(gs[1, 0])
ax3.hist(cf_accrual['net_cashflow'], bins=40, alpha=0.55, color='darkgreen',
         label=f"Accrual (median ₹{s_accrual['median']:,.0f})", edgecolor='white')
ax3.hist(cf_projected['net_cashflow'], bins=40, alpha=0.55, color='navy',
         label=f"Payment-projected (median ₹{s_projected['median']:,.0f})", edgecolor='white')
ax3.axvline(0, color='black', linestyle='--', linewidth=0.8, alpha=0.6)
ax3.set_xlabel('Weekly net cashflow (INR)')
ax3.set_ylabel('Week count')
ax3.set_title('Distribution of weekly net cashflow — accrual vs payment-projected', fontsize=11)
ax3.legend(loc='upper left', fontsize=9)
ax3.grid(True, alpha=0.3)

# Panel 4 — Cumulative comparison
ax4 = fig.add_subplot(gs[1, 1])
ax4.plot(cf_accrual.index, cf_accrual['net_cashflow'].cumsum(),
         color='darkgreen', linewidth=1.2, label='Accrual cumulative')
ax4.plot(cf_projected.index, cf_projected['net_cashflow'].cumsum(),
         color='navy', linewidth=1.2, label='Payment-projected cumulative')
ax4.axhline(0, color='black', linestyle='--', linewidth=0.6, alpha=0.5)
ax4.set_ylabel('Cumulative net cashflow (INR)')
ax4.set_title('Cumulative net cashflow — full 3 years', fontsize=11)
ax4.legend(loc='upper left', fontsize=9)
ax4.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
ax4.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.setp(ax4.xaxis.get_majorticklabels(), rotation=45, ha='right')
ax4.grid(True, alpha=0.3)

# Summary table
ax5 = fig.add_subplot(gs[2, :])
ax5.axis('off')

table_data = [
    ['Metric',              s_accrual['label'],                  s_projected['label']],
    ['Weeks in series',     f"{s_accrual['weeks']}",             f"{s_projected['weeks']}"],
    ['Mean (INR)',          f"{s_accrual['mean']:,.0f}",         f"{s_projected['mean']:,.0f}"],
    ['Median (INR)',        f"{s_accrual['median']:,.0f}",       f"{s_projected['median']:,.0f}"],
    ['Std deviation (INR)', f"{s_accrual['std']:,.0f}",          f"{s_projected['std']:,.0f}"],
    ['Deficit weeks (%)',   f"{s_accrual['deficit_pct']:.1f}%",  f"{s_projected['deficit_pct']:.1f}%"],
    ['Min (INR)',           f"{s_accrual['min']:,.0f}",          f"{s_projected['min']:,.0f}"],
    ['Max (INR)',           f"{s_accrual['max']:,.0f}",          f"{s_projected['max']:,.0f}"],
]

table = ax5.table(cellText=table_data, loc='center', cellLoc='center',
                  colWidths=[0.32, 0.34, 0.34])
table.auto_set_font_size(False)
table.set_fontsize(9.5)
table.scale(1, 1.6)
for i in range(3):
    cell = table[(0, i)]
    cell.set_facecolor('#2C3E50')
    cell.set_text_props(color='white', weight='bold')

fig.suptitle('Phase 2 output — SME weekly cashflow, two target constructions',
             fontsize=13, y=0.995)

os.makedirs("../reports/figures", exist_ok=True)
plt.savefig("../reports/figures/phase2_summary.png", bbox_inches='tight', dpi=150)
plt.show()

print("\nPhase 2 summary figure saved to reports/figures/phase2_summary.png")

## Phase 2 methodology, summary and decisions

**Two forecasting targets constructed:**
- **Accrual target** (`weekly_cashflow.parquet`): weekly aggregation by invoice-issue 
  date. 158 weeks. Standard SME accounting-basis measure.
- **Payment-projected target** (`weekly_cashflow_projected.parquet`): each invoice 
  projected to expected payment week using per-customer contractual terms. 163 weeks. 
  Cash-basis proxy.

**Payment-projection methodology:**
- Per-customer payment terms collected from SME accountant: 100% coverage of 74 buyers, 
  96.4% of purchase invoices (18 unmapped suppliers assigned a weighted-average default 
  of ~20 days).
- 2-day banking delay applied to every non-immediate term.
- Size-conditioning for range terms: each buyer's own 75th-percentile invoice value 
  used as the "large" threshold, reflecting the accountant's observation that size 
  perception varies by customer. 103 of 543 sales invoices reclassified vs a fixed 
  22,500 rupee rule.

**Correlation diagnostic:**
- Payment-cycle features show near-zero linear correlation with both targets 
  (Pearson |r| < 0.10 across all variants).
- Relationship is time-varying: 12-week rolling correlation ranges from -0.7 to +0.55 
  across the observation window.
- Interpretation: aggregate multi-week features cannot capture single-week net cashflow 
  variance linearly. Features retained as candidate predictors evaluated empirically 
  via SHAP in Phase 3, rather than assumed a priori.

**Both targets and full feature set retained for Phase 3 modelling.**

---

**Output artefacts (saved to `data/processed/`):**
- `sales_clean.parquet` and `purchase_clean.parquet`: cleaned, pseudonymised invoice data
- `weekly_cashflow.parquet`: accrual target
- `weekly_cashflow_projected.parquet`: payment-projected target
- `sales_with_projections.parquet` and `purchase_with_projections.parquet`: per-invoice projections
- `features_v1.parquet`: engineered features (calendar, lag, rolling, payment-cycle)

**Summary figure:** `reports/figures/phase2_summary.png`

**Phase 3 begins with `src/` refactor and ARIMA baseline against both targets.**